# 🦙 DigiFair AI — Llama / LoRA Training with W&B

این نوت‌بوک برای اضافه کردن فرایند آموزش/فاین‌تیون به پروژه Exhibition RAG ساخته شده است.

✅ ویژگی‌ها:
- ساخت دیتاست آموزشی از شرکت‌های نمایشگاه
- آموزش سبک با LoRA/QLoRA
- لاگ کردن loss و metric در Weights & Biases
- ذخیره adapter خروجی
- امکان push به Hugging Face Hub

⚠️ نکته امنیتی:
- `WANDB_API_KEY` و `HF_TOKEN` را فقط در Colab Secrets بگذارید.
- هیچ توکنی را داخل سلول‌ها hard-code نکنید.


## 0) انتخاب مدل

اگر به مدل‌های رسمی Meta Llama دسترسی داری، می‌توانی `MODEL_ID` را روی یکی از مدل‌های Llama بگذاری.
اگر دسترسی نداری، برای تست آموزشی از TinyLlama استفاده کن.


In [ ]:
# برای تست سبک و عمومی:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# اگر HF_TOKEN و دسترسی Llama داری، می‌توانی یکی از این‌ها را امتحان کنی:
# MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

PROJECT_NAME = "exhibition-rag-llama-training"
RUN_NAME = "digifair-lora-run"
OUTPUT_DIR = "/content/digifair-lora-adapter"


## 1) نصب کتابخانه‌ها

In [ ]:
!pip install -q -U transformers accelerate datasets peft trl bitsandbytes wandb huggingface_hub pandas requests scikit-learn


## 2) اتصال امن به Hugging Face و W&B

In [ ]:
import os
from getpass import getpass

def get_secret(name, prompt):
    # Colab Secrets first
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # Environment
    val = os.environ.get(name)
    if val:
        return val
    # Manual input
    return getpass(prompt)

HF_TOKEN = get_secret("HF_TOKEN", "HF_TOKEN را وارد کنید یا Enter بزنید اگر لازم نیست: ")
WANDB_API_KEY = get_secret("WANDB_API_KEY", "WANDB_API_KEY را وارد کنید یا Enter بزنید اگر نمی‌خواهید W&B فعال شود: ")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ["WANDB_PROJECT"] = PROJECT_NAME
    os.environ["WANDB_NAME"] = RUN_NAME

print("HF token:", bool(HF_TOKEN))
print("W&B token:", bool(WANDB_API_KEY))


## 3) دانلود دیتای شرکت‌ها

In [ ]:
import requests, pandas as pd, json, re

COMPANIES_JSON_URL = "https://sosa123456-exhibition-connector-rag2-static.static.hf.space/data/companies.json"
resp = requests.get(COMPANIES_JSON_URL, timeout=60)
resp.raise_for_status()
companies = resp.json()
print("records:", len(companies))
companies[0]


## 4) ساخت دیتاست Instruction برای Llama

In [ ]:
def clean(x):
    return str(x or "").strip()

def company_answer(r):
    return f"""نام شرکت: {clean(r.get('company'))}
زمینه فعالیت: {clean(r.get('activity')) or 'نامشخص'}
زمینه کاری: {clean(r.get('category')) or 'نامشخص'}
وب‌سایت: {clean(r.get('website')) or 'ثبت نشده'}
سالن/غرفه: {clean(r.get('hall')) or 'ثبت نشده'} / {clean(r.get('booth')) or 'ثبت نشده'}"""

examples = []
for r in companies:
    c = clean(r.get('company'))
    if not c or c == 'بدون نام':
        continue
    examples.append({
        "instruction": f"اطلاعات شرکت {c} را در نمایشگاه توضیح بده.",
        "output": company_answer(r)
    })
    if r.get('activity'):
        examples.append({
            "instruction": f"کدام شرکت در زمینه {clean(r.get('activity'))[:80]} فعالیت می‌کند؟",
            "output": company_answer(r)
        })
    if r.get('hall'):
        examples.append({
            "instruction": f"کدام شرکت در سالن {clean(r.get('hall'))} حضور دارد؟",
            "output": company_answer(r)
        })

# محدود کردن برای آموزش سریع دمو
examples = examples[:2500]
print("training examples:", len(examples))
pd.DataFrame(examples).head()


In [ ]:
from datasets import Dataset

def format_prompt(row):
    return f"""<|system|>
شما دستیار فارسی نمایشگاه هستید. فقط بر اساس داده‌های نمایشگاه پاسخ بدهید.
<|user|>
{row['instruction']}
<|assistant|>
{row['output']}"""

dataset = Dataset.from_list([{"text": format_prompt(x)} for x in examples])
dataset = dataset.train_test_split(test_size=0.05, seed=42)
dataset


## 5) بارگذاری مدل و آماده‌سازی LoRA

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

use_cuda = torch.cuda.is_available()
print("CUDA:", use_cuda, torch.cuda.get_device_name(0) if use_cuda else "CPU")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or None, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN or None, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, token=HF_TOKEN or None, trust_remote_code=True)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"] if "llama" in MODEL_ID.lower() else ["q_proj", "v_proj"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6) آموزش با W&B Logging

In [ ]:
import wandb
from trl import SFTTrainer, SFTConfig

report_to = []
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    wandb.init(project=PROJECT_NAME, name=RUN_NAME, config={"model": MODEL_ID, "examples": len(examples)})
    report_to = ["wandb"]

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1 if use_cuda else 1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    max_steps=80,  # برای دمو سریع؛ برای آموزش واقعی بیشتر کن
    logging_steps=5,
    save_steps=40,
    eval_steps=40,
    report_to=report_to,
    fp16=use_cuda,
    bf16=False,
    max_seq_length=1024,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    dataset_text_field="text",
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
if WANDB_API_KEY:
    wandb.finish()
print("saved to", OUTPUT_DIR)


## 7) تست مدل آموزش‌دیده

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=180)
prompt = """<|system|>
شما دستیار فارسی نمایشگاه هستید.
<|user|>
وب‌سایت شرکت پریسماتک چیست؟
<|assistant|>
"""
out = pipe(prompt)[0]["generated_text"]
print(out)


## 8) آپلود Adapter به Hugging Face Hub اختیاری

In [ ]:
# اختیاری: اگر می‌خواهی adapter را روی Hugging Face ذخیره کنی
# از Colab Secrets یک HF_TOKEN با write permission لازم است.

PUSH_TO_HUB = False
REPO_ID = "YOUR_USERNAME/digifair-rag-lora"

if PUSH_TO_HUB:
    model.push_to_hub(REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
    print("pushed to", REPO_ID)
else:
    print("Push disabled. Adapter saved locally at", OUTPUT_DIR)


## 9) اتصال نتیجه آموزش به سایت

در نسخه واقعی:
1. Adapter یا مدل را روی Hugging Face ذخیره کنید.
2. در Vercel Env این‌ها را بگذارید:
   - `HF_TOKEN`
   - `HF_MODEL` یا `TRAINED_ADAPTER_REPO`
   - `WANDB_API_KEY`
3. Endpoint پاسخ‌دهی می‌تواند به مدل آموزش‌دیده وصل شود.

فعلاً سایت production از Vercel JSON RAG استفاده می‌کند و این نوت‌بوک برای آموزش/آزمایش مدل است.


## 10) تست Gradio فارسی RTL

این سلول برای تست نمایش صحیح فارسی در Gradio است.


In [ ]:
!pip install -q gradio
import gradio as gr

def format_fa_answer(text):
    text = str(text or "").strip()
    # جداسازی ساده خطوط و بهبود خوانایی فارسی
    text = text.replace("|", "\n- ")
    return text

def demo_answer(message, history):
    # اگر مدل pipe ساخته شده باشد، از آن استفاده می‌کنیم؛ وگرنه پاسخ دمو
    try:
        prompt = f"""<|system|>
شما دستیار فارسی نمایشگاه هستید. پاسخ کوتاه، دقیق و راست‌چین بنویسید.
<|user|>
{message}
<|assistant|>
"""
        out = pipe(prompt)[0]["generated_text"]
        ans = out.split("<|assistant|>")[-1].strip()
    except Exception:
        ans = "نمونه پاسخ فارسی:\n- شرکت پریسماتک در حوزه ابزار دقیق فعالیت دارد.\n- وب‌سایت: https://prismatech.ir/\n- اگر سالن/غرفه در اکسل ثبت نشده باشد، باید در پنل ادمین تکمیل شود."
    return format_fa_answer(ans)

css = """
.gradio-container { direction: rtl; font-family: Tahoma, Arial, sans-serif; }
textarea, input, .prose, .markdown, .message { direction: rtl !important; text-align: right !important; }
footer { display: none !important; }
"""

demo = gr.ChatInterface(
    fn=demo_answer,
    title="دستیار فارسی نمایشگاه — تست Gradio RTL",
    description="برای تست نوشتار فارسی، راست‌چین و پاسخ مدل آموزش‌دیده/دمو",
    examples=["وب‌سایت شرکت پریسماتک چیست؟", "مواد شیمیایی تصفیه آب", "سالن 31B"],
    css=css,
    chatbot=gr.Chatbot(rtl=True, height=420, show_copy_button=True),
    textbox=gr.Textbox(placeholder="سؤال خود را فارسی بنویسید...", rtl=True),
)

demo.launch(share=True, debug=True)
